# VoiceShield (BAFV-PCTA) — Phase 7 Training, Calibration & Evaluation

Run this notebook on **Google Colab (Runtime -> Change runtime type -> T4 GPU)** to:
1. Train real anti-spoofing weights (`models/aasist_baseline.pt`) with 8 kHz telephony augmentation.
2. Calibrate empirical BAFV & Open-Set Generator centroids (`models/calibration_stats.json`).
3. Run the Phase 7 benchmark evaluation (`EER`, `min-tDCF`, `ROC-AUC`, Telephony Guard ablation, and LOGO Open-Set).
4. Download the trained artifacts ready to commit to your GitHub repo (`models/`).

In [ ]:
#@title 1. Check GPU & Clone VoiceShield Repository
!nvidia-smi
%cd /content
!rm -rf Capstone-Project
!git clone https://github.com/Rahul1062005/Capstone-Project.git
%cd /content/Capstone-Project
!pip install -q -r requirements.txt
import torch
print('PyTorch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())

## 2. Prepare Your Dataset
Choose **one** of the options below:
- **Option A (Recommended):** Download **ASVspoof 2019 LA** via Kaggle (`kaggle.json`) or mount Google Drive.
- **Option B:** Export a folder dataset into `/content/dataset/bonafide/*.wav` and `/content/dataset/spoof/*.wav` (e.g., from In-the-Wild or WaveFake).

In [ ]:
#@title 2A. Option A — Mount Google Drive (if you stored ASVspoof / In-the-Wild in Drive)
from google.colab import drive
drive.mount('/content/drive')

# Update paths below to match your Drive or /content folder:
DATASET_MODE = 'folder'  # 'folder' or 'asvspoof'
DATA_DIR = '/content/drive/MyDrive/voiceshield_dataset'  # expects bonafide/ and spoof/

# For ASVspoof 2019 LA mode:
ASV_TRAIN_PROTO = '/content/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt'
ASV_TRAIN_AUDIO = '/content/LA/ASVspoof2019_LA_train/flac'
ASV_DEV_PROTO = '/content/LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt'
ASV_DEV_AUDIO = '/content/LA/ASVspoof2019_LA_dev/flac'

In [ ]:
#@title 3. Train Baseline Anti-Spoofing Detector (saves models/aasist_baseline.pt)
EPOCHS = 20 #@param {type:'integer'}
BATCH_SIZE = 32 #@param {type:'integer'}
TELEPHONY_AUG_PROB = 0.25 #@param {type:'number'}

if DATASET_MODE == 'folder':
    !python scripts/train_detector.py \
        --data-dir "$DATA_DIR" \
        --epochs $EPOCHS \
        --batch-size $BATCH_SIZE \
        --telephony-aug-prob $TELEPHONY_AUG_PROB \
        --output-checkpoint models/aasist_baseline.pt \
        --metrics-out models/training_metrics.json
else:
    !python scripts/train_detector.py \
        --asvspoof-protocol "$ASV_TRAIN_PROTO" \
        --audio-root "$ASV_TRAIN_AUDIO" \
        --dev-protocol "$ASV_DEV_PROTO" \
        --dev-audio-root "$ASV_DEV_AUDIO" \
        --epochs $EPOCHS \
        --batch-size $BATCH_SIZE \
        --telephony-aug-prob $TELEPHONY_AUG_PROB \
        --output-checkpoint models/aasist_baseline.pt \
        --metrics-out models/training_metrics.json

In [ ]:
#@title 4. Calibrate BAFV Coupling & Open-Set Generator Centroids
BONAFIDE_DIR = f"{DATA_DIR}/bonafide"
SPOOF_BY_GEN_DIR = ''  # Optional: path with subfolders per generator (e.g. ElevenLabs/, RVC/, Bark/)

if SPOOF_BY_GEN_DIR:
    !python scripts/calibrate_manifold.py \
        --bonafide-dir "$BONAFIDE_DIR" \
        --spoof-root "$SPOOF_BY_GEN_DIR" \
        --checkpoint models/aasist_baseline.pt \
        --output models/calibration_stats.json \
        --max-files 500
else:
    !python scripts/calibrate_manifold.py \
        --bonafide-dir "$BONAFIDE_DIR" \
        --checkpoint models/aasist_baseline.pt \
        --output models/calibration_stats.json \
        --max-files 500

In [ ]:
#@title 5. Run Phase 7 Evaluation (Clean vs. 8 kHz Telephony + LOGO + min-tDCF)
MAX_EVAL_SAMPLES = 1000 #@param {type:'integer'}

if DATASET_MODE == 'folder':
    extra = f'--spoof-root "{SPOOF_BY_GEN_DIR}"' if SPOOF_BY_GEN_DIR else ''
    !python scripts/evaluate_phase7.py \
        --data-dir "$DATA_DIR" \
        --checkpoint models/aasist_baseline.pt \
        --calibration models/calibration_stats.json \
        --max-samples $MAX_EVAL_SAMPLES \
        $extra
else:
    !python scripts/evaluate_phase7.py \
        --asvspoof-protocol "$ASV_DEV_PROTO" \
        --audio-root "$ASV_DEV_AUDIO" \
        --checkpoint models/aasist_baseline.pt \
        --calibration models/calibration_stats.json \
        --max-samples $MAX_EVAL_SAMPLES

!cat docs/PHASE_7_EVALUATION_REPORT.md

In [ ]:
#@title 6. Download Trained Weights & Phase 7 Report
from google.colab import files

for artifact in [
    'models/aasist_baseline.pt',
    'models/training_metrics.json',
    'models/calibration_stats.json',
    'docs/PHASE_7_EVALUATION_REPORT.md',
    'data/predictions/phase7_eval.json',
]:
    try:
        files.download(artifact)
    except Exception as exc:
        print('Skipped:', artifact, exc)